# Remote Deployment with Hetzner Cloud

This notebook demonstrates deploying a netrun pool server to a cloud VM and running
a network that offloads computation to it.

Each step is a separate cell so you can iterate independently:
1. **Create** a Hetzner Cloud server (idempotent)
2. **Deploy** the `app/` folder (upload code + install deps + auto-delete watchdog)
3. **Start** the pool server + SSH tunnel
4. **Run** a network with a remote pool
5. **Stop** the pool server and close the SSH tunnel

The server will **automatically delete itself** after being idle for 10 minutes
(no pool server running), thanks to a systemd watchdog installed during deployment.
This prevents forgotten servers from accumulating charges.

**Prerequisites:**
- `hcloud` CLI installed and authenticated (`hcloud context create`)
- An SSH key registered in your Hetzner Cloud project (`hcloud ssh-key list`)
- The corresponding private key available locally
- A `.env` file in this directory (copy `.env.example` and fill in your values)

## Configuration

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

# --- Loaded from .env (see .env.example) ---
HCLOUD_SSH_KEY_NAME = os.environ["HCLOUD_SSH_KEY_NAME"]
SSH_PRIVATE_KEY_PATH = os.environ["SSH_PRIVATE_KEY_PATH"]
HCLOUD_API_TOKEN = os.environ["HCLOUD_API_TOKEN"]

# --- Server settings ---
SERVER_NAME = "netrun-demo"
SERVER_TYPE = "cpx22"          # 2 vCPU, 4 GB RAM
SERVER_IMAGE = "ubuntu-24.04"
SERVER_LOCATION = "fsn1"       # Falkenstein, DE

# --- Deployment settings ---
REMOTE_DIR = "/opt/netrun-app"
POOL_SERVER_PORT = 8765
APP_DIR = str(Path("./app").resolve())

## Create Server

Creates a Hetzner Cloud VM. Idempotent — if the server already exists, returns its IP immediately.

In [2]:
from deploy_to_hetzner import create_hetzner_server

ip = create_hetzner_server(
    server_name=SERVER_NAME,
    ssh_key_name=HCLOUD_SSH_KEY_NAME,
    server_type=SERVER_TYPE,
    server_image=SERVER_IMAGE,
    server_location=SERVER_LOCATION,
)
print(f"Server IP: {ip}")

Server 'netrun-demo' already exists at 46.224.130.14
Server IP: 46.224.130.14


## Deploy

Uploads the `app/` folder, installs dependencies with `uv`, and writes the server scripts.
Also installs the **auto-delete watchdog** — a systemd timer that deletes the server
if the pool server hasn't been running for 10 minutes. Skips if already deployed.

In [ ]:
from deploy_to_hetzner import check_deployed, deploy_to_server

if not check_deployed(host=ip, ssh_private_key_path=SSH_PRIVATE_KEY_PATH, remote_dir=REMOTE_DIR):
    deploy_to_server(
        host=ip,
        ssh_private_key_path=SSH_PRIVATE_KEY_PATH,
        local_folder=APP_DIR,
        remote_dir=REMOTE_DIR,
        net_source="netrun.toml",
        pool_server_port=POOL_SERVER_PORT,
        python_version="3.11",
        pre_commands=["apt-get update -qq && apt-get install -y -qq build-essential > /dev/null 2>&1"],
        exclude=["*.pyc"],
        exclude_dir=["__pycache__", "*/__pycache__", ".venv"],
        enable_watchdog=True,
        hcloud_api_token=HCLOUD_API_TOKEN,
        auto_delete_idle_minutes=10,
        auto_delete_start_delay_minutes=15,
    )
else:
    print("Already deployed — skipping.")

## Start Pool Server

Launches the pool server on the remote and opens an SSH tunnel so the client can reach it via `ws://localhost:8765`.

In [ ]:
from deploy_to_hetzner import check_pool_server_running

check_pool_server_running(
    host=ip,
    ssh_private_key_path=SSH_PRIVATE_KEY_PATH,
    remote_dir=REMOTE_DIR,
)

False

In [ ]:
from deploy_to_hetzner import start_pool_server

handle = start_pool_server(
    host=ip,
    ssh_private_key_path=SSH_PRIVATE_KEY_PATH,
    remote_dir=REMOTE_DIR,
    pool_server_port=POOL_SERVER_PORT,
)
print(f"Pool: {handle.pool_server_url}")

Opening SSH tunnel (localhost:8765 -> 188.245.171.241:8765)...
  Tunnel active (pid 36785)
Waiting for remote port 8765. ready!
Pool server started: ws://localhost:8765
Pool: ws://localhost:8765


## Run the Network

The graph, pools, edges, and output queues are defined in `app/net_config.toml`.
Only the remote pool's `url` and `worker_name` are set at runtime since they
depend on the SSH tunnel.

The network flow is:
```
double(x=5) → 10 → add(a=10, b=10) → 20 → format_result(value=20) → "The answer is: 20"
```

All three nodes execute on the remote server. Packets flow between them
on the remote, and we collect the final result locally via an output queue.

In [ ]:
import sys

# Add app/ to path so the client can resolve the function factory
sys.path.insert(0, APP_DIR)

from netrun.net import Net
from netrun.net.config import NetConfig

# Load the full config from TOML (graph, pools, edges, output queues)
config = NetConfig.from_file(f"{APP_DIR}/netrun.toml")

# Set the runtime-dependent fields (url depends on the SSH tunnel)
config.pools["remote"].spec.url = handle.pool_server_url
config.pools["remote"].spec.worker_name = "execution_manager"

async with Net(config) as net:
    # Inject input data
    net.inject_data("double", "x", [5])
    net.inject_data("add", "b", [10])

    # Run until all processing is complete
    await net.run_until_blocked()

    # Retrieve results from the output queue
    results = net.flush_output_queue("results")
    print("Result:")
    print(results[0])

    await net.request_pool_shutdown("remote")

## Logs

In [ ]:
net.logs.print_all()

## Stop

Close the SSH tunnel and stop the remote pool server. The auto-delete watchdog will
delete the server after 10 minutes of idle time — no manual cleanup needed.

In [ ]:
from deploy_to_hetzner import stop_pool_server

handle.close_tunnel()
stop_pool_server(host=ip, ssh_private_key_path=SSH_PRIVATE_KEY_PATH, remote_dir=REMOTE_DIR)

SSH tunnel closed.
No pool server running.


Uncomment the below if you want to delete the server manually.

In [ ]:
# from deploy_to_hetzner import delete_server
# delete_server("netrun-demo")

Deleting server 'netrun-demo'...
  Done.
